# Script 7 - Rhythm metrics to mediators, a-paths (Supplementary Table S14)

Estimates the standardised rhythm -> KDM-Biological Age advancement and
rhythm -> log Systemic Immune-Inflammation Index associations for each of
intradaily variability (IV), interdaily stability (IS) and relative amplitude
(RA), using survey-weighted multivariable linear regression (WLS) with HC3
standard errors.

Each regression uses the per-outcome sample (exposure, outcome, weight and
covariates non-missing), matching the FIML treatment of the Y-variables in
Mplus, where each equation effectively uses its own sample. The exposure and the
outcome are z-standardised with the survey-weighted mean and SD computed on that
sample, which is how Mplus STDYX rescales coefficients under TYPE = COMPLEX +
MLR; standardising with unweighted reference-sample SDs leaves a residual
STDYX discrepancy of about 0.01.

Input: `data/processed/8_nhanes_analytic.csv`. Outputs:
`result/supplementary/SuppTableS14_APaths_WLS.csv` and `.xlsx`,
`result/raw/script7_a_paths_full.csv`.


## Step 0 - Imports and paths


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

import os as _os

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = _os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE = find_root()
PROC   = BASE / 'data' / 'processed'
R_MAIN = BASE / 'result' / 'main'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
for p in (R_MAIN, R_SUPP, R_RAW):
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook uses the code on disk.
import importlib
import wls_utils, nature_palette, mplus_parse
for _m in (nature_palette, wls_utils, mplus_parse,):
    importlib.reload(_m)

from wls_utils import wls_fit

## Step 1 - Load 8_nhanes_analytic.csv and rename columns


In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv')
print(f'Loaded 8_nhanes_analytic.csv: {df.shape}')

rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
    'obj_mvpa_min':   'OBJ_MVPA',
    'SDMVSTRA':       'STRATUM',
    'SDMVPSU':        'PSU',
    'KDM_BA_advance': 'KDM_BA_ADV',
    'SII_log':        'SII_LOG',
    'dpq_sum':        'PHQ9',
}
df = df.rename(columns=rename)
# WTMEC4YR is stored in the analytic file; checked here, not re-derived.
assert 'WTMEC4YR' in df.columns, 'WTMEC4YR missing from the analytic file'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= 18].copy()
print(f'Adults 18+: n = {len(df):,}')

## Step 2 - Helpers: per-outcome sample and survey-weighted mean / SD

M1 covariates: age, sex, BMI, education, wear time, sleep duration. M2 adds
total daily activity volume (TOTAL_MIMS).


In [ ]:
M1_COVS  = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']

# M2 adds total daily activity volume (TOTAL_MIMS). TOTAL_MIMS and
# threshold-MVPA correlate 0.903 and give near-identical adjustments (a1 for
# IV: +0.046 unadjusted, +0.027 with MVPA, +0.032 with TOTAL_MIMS; IS -0.096 /
# -0.069 / -0.071; RA -0.116 / -0.110 / -0.120). TOTAL_MIMS needs no cut-point;
# MIMS has no consensus MVPA cut-point (45.1 gives 22 min/day, 19.614 gives
# 249). M10 is not used as the volume covariate: it correlates 0.937 with
# total volume and is part of RA = (M10-L5)/(M10+L5), so conditioning on it
# conditions on part of the exposure (a1 for IV falls to +0.010, p = 0.43).
M2_EXTRA = ['TOTAL_MIMS']
from nature_palette import EXP_ORDER
from nature_palette import fmt_ci as _fmt_ci, age_label
EXPOSURES = list(EXP_ORDER)   # RA, IS, IV

def per_outcome_sample(outcome_col, extra_required=(), exposure='IV'):
    needed = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS + list(extra_required)
    mask = df[needed].notna().all(axis=1)
    return df.loc[mask].copy()

def weighted_mean_sd(x, w):
    """Survey-weighted mean and SD (denominator = sum of weights, no Bessel correction, as in Mplus TYPE = COMPLEX)."""
    w_tot = np.sum(w)
    mu    = np.sum(x * w) / w_tot
    var   = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

print('Per-outcome sample sizes:')
for col, label in [('KDM_BA_ADV', 'KDM-BA advancement'),
                   ('SII_LOG',    'log SII           ')]:
    n_m1 = len(per_outcome_sample(col))
    n_m2 = len(per_outcome_sample(col, M2_EXTRA))
    print(f'  {label}   M1 n = {n_m1:,}   M2 (+MVPA) n = {n_m2:,}')
# Mplus equation-level n is read from the .out files, not transcribed here.

## Step 3 - WLS-HC3 with sample-specific survey-weighted standardisation

For one mediator:

1. Filter to the per-outcome sample (exposure, outcome, weight and covariates non-null).
2. Compute the survey-weighted mean and SD of the exposure and the outcome on that sample.
3. Z-standardise the exposure and the outcome with those weighted SDs.
4. Fit survey-weighted WLS with HC3 SE on the standardised values; covariates stay on their raw scale.
5. The exposure coefficient is the STDYX beta, comparable to Mplus.


In [ ]:
def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def run_a_path_stdyx(outcome_col, covariates, extra_required=(), iv_col='IV'):
    ana = per_outcome_sample(outcome_col, extra_required, exposure=iv_col)
    w   = ana['WTMEC4YR'].values
    # Survey-weighted standardisation of exposure and outcome on this sample
    mu_iv,  sd_iv  = weighted_mean_sd(ana[iv_col].values,      w)
    mu_out, sd_out = weighted_mean_sd(ana[outcome_col].values, w)
    iv_z      = (ana[iv_col].values      - mu_iv)  / sd_iv
    out_z     = (ana[outcome_col].values - mu_out) / sd_out
    # Design matrix; covariates on raw scale (Mplus STDYX convention)
    X = np.column_stack([iv_z] + [ana[c].values for c in covariates])
    fit = wls_fit(X, out_z, w=w, se='hc3', add_const=True)
    return {
        'exposure':    iv_col,
        'outcome':     outcome_col,
        'covariates':  ', '.join(covariates),
        'n':           int(fit['n']),
        'k':           int(fit['k']),
        'sd_w_iv':     sd_iv,
        'sd_w_out':    sd_out,
        'beta':        float(fit['params'][1]),
        'se':          float(fit['se'][1]),
        't':           float(fit['t'][1]),
        'p':           float(fit['p'][1]),
        'ci_lo':       float(fit['ci_lo'][1]),
        'ci_hi':       float(fit['ci_hi'][1]),
        'r2_model':    float(fit['r2']),
    }

## Step 4 - Run M1 (core) and M2 (+ total volume) for each exposure and mediator


In [ ]:
results = []
for exp in EXPOSURES:
    for outcome_label, outcome_col in [
        ('KDM-BA advancement', 'KDM_BA_ADV'),
        ('log SII',            'SII_LOG'),
    ]:
        for model_label, covs, req in [
            ('M1 (core)',            M1_COVS,            ()),
            ('M2 (+ total volume)',  M1_COVS + M2_EXTRA, M2_EXTRA),
        ]:
            r = run_a_path_stdyx(outcome_col, covs, extra_required=req, iv_col=exp)
            r['outcome_label'] = outcome_label
            r['model_label']   = model_label
            results.append(r)
            print(f"{exp:3s} -> {outcome_label:20s} {model_label:22s} n={r['n']:,}  "
                  f"beta={r['beta']:+.3f} ({r['ci_lo']:+.3f}, {r['ci_hi']:+.3f})  "
                  f"p={fmt_p(r['p'])}")
    print()

results_raw = pd.DataFrame(results)
results_raw.to_csv(R_RAW / 'script7_a_paths_full.csv', index=False)
print(f'\nWrote {R_RAW / "script7_a_paths_full.csv"}')

## Step 5 - Build Supplementary Table S14

One block per exposure and mediator; rows M1 and M2 with n, STDYX beta (95% CI), p and R2.


In [ ]:
rows = []
for exp in EXPOSURES:
  for outcome_label in ['KDM-BA advancement', 'log SII']:
    block = results_raw[(results_raw['outcome_label'] == outcome_label) &
                        (results_raw['exposure'] == exp)].reset_index(drop=True)
    rows.append({'Outcome': f'{exp} -> {outcome_label}', 'Model': '', 'n': '',
                 'beta_STDYX (95% CI)': '', 'p': '', 'R2': ''})
    for _, r in block.iterrows():
        rows.append({
            'Outcome':             '',
            'Model':               r['model_label'],
            'n':                   f"{int(r['n']):,}",
            'beta_STDYX (95% CI)': fmt_ci(r['beta'], r['ci_lo'], r['ci_hi']),
            'p':                   fmt_p(r['p']),
            'R2':                  f"{r['r2_model']:.3f}",
        })

table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))
table2.to_csv(R_SUPP / 'SuppTableS14_APaths_WLS.csv', index=False)
try:
    with pd.ExcelWriter(R_SUPP / 'SuppTableS14_APaths_WLS.xlsx', engine='openpyxl') as xw:
        table2.to_excel(xw, sheet_name='Supp Table S14', index=False)
        pd.DataFrame({'Notes': [
            'Survey-weighted multivariable linear regression (WTMEC4YR = WTMEC2YR / 2).',
            'Source: data/processed/8_nhanes_analytic.csv.',
            'Sample: all adults aged 18 and over, NOT stratified by age. The primary analysis in Table 2 is fitted within three age-and-outcome cells; this table pools them. Pooled and stratified estimates differ where the association is age-dependent -- for intradaily variability they differ in sign -- so read this table as a supportive full-adult fit, not as a version of Table 2.',
            'Per-outcome analytic sample: IV + outcome + M1 covariates non-null (other mediator and PHQ-9 not required, mirroring Mplus FIML treatment of Y).',
            'IV and outcome z-standardised using SURVEY-WEIGHTED mean and SD on the per-outcome sample (Mplus convention under TYPE = COMPLEX + MLR).',
            'Reported beta = STDYX standardised IV -> mediator association (per 1 SD of IV, in mediator SDs).',
            'Standard errors: HC3 heteroscedasticity-consistent (MacKinnon-White) via wls_utils.wls_fit.',
            'M1 and M2 name covariate sets (core; core plus total activity volume), not Mplus model files.',
            'M1 (core): age, sex, body mass index, education, accelerometer wear-time, self-reported sleep duration.',
            'M2 (+ total volume): M1 + total daily MIMS. Volume is the confounder for any rhythm claim; MVPA is not used because MIMS has no consensus cut-point.',
            'These are supportive fits. Weights enter the point estimate but the variance carries no stratum or PSU adjustment, so the standard errors are smaller than the design-based ones. Quote Mplus for inference.',
        ]}).to_excel(xw, sheet_name='Notes', index=False)
    print(f'Wrote {R_SUPP / "SuppTableS14_APaths_WLS.xlsx"}')
except Exception as e:
    print(f'(xlsx skipped: {e})')

# Move superseded result/main/Table2_a_paths files, if present, to _superseded/.
for _stale in [R_MAIN / 'Table2_a_paths.csv', R_MAIN / 'Table2_a_paths.xlsx']:
    if _stale.exists():
        _dest = BASE / '_superseded' / _stale.name
        _dest.parent.mkdir(exist_ok=True)
        _stale.replace(_dest)
        print(f'superseded -> _superseded/{_stale.name}')


## Step 6 - Consistency check against Mplus SEM a-paths

Compares the IV M1 STDYX coefficients with those parsed from
`9_mplus_models/M1_depression.out`; the check is same sign and |delta| < 0.03.


In [ ]:
# Self-check against Mplus: STDYX a-paths parsed from M1_depression.out at run time.
from mplus_parse import parse_file

_est, _eff, _info = parse_file(BASE / 'scripts' / '9_mplus_models' / 'M1_depression.out')
_est = pd.DataFrame(_est)
_std = _est[_est.solution == 'STDYX']

def _mplus_coef(section, param):
    r = _std[(_std.section == section) & (_std.parameter == param)]
    return float(r.iloc[0].estimate) if len(r) else np.nan

MPLUS_REF = {
    'KDM-BA advancement': _mplus_coef('BA_ADV ON', 'IV'),
    'log SII':            _mplus_coef('SII_LOG ON', 'IV'),
}
print('Self-check: WLS-HC3 STDYX (IV, M1) vs Mplus M1_depression.out, parsed')
print('-' * 78)
for outcome_label, mplus_beta in MPLUS_REF.items():
    m1 = results_raw[(results_raw['outcome_label'] == outcome_label)
                     & (results_raw['exposure'] == 'IV')
                     & (results_raw['model_label'] == 'M1 (core)')].iloc[0]
    d = m1['beta'] - mplus_beta
    # The estimators differ (HC3 vs MLR+FIML with the survey correction);
    # the check is same sign and |delta| < 0.03.
    same_sign = np.sign(m1['beta']) == np.sign(mplus_beta)
    flag = 'OK' if (same_sign and abs(d) < 0.03) else 'REVIEW'
    print(f"  {outcome_label:22s}  WLS={m1['beta']:+.3f}  Mplus={mplus_beta:+.3f}  "
          f"delta={d:+.3f}  [{flag}]")
    assert same_sign, f'{outcome_label}: WLS and Mplus disagree in SIGN -- stop and look'
